Import des libraires

In [1]:
import os
import time
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
import cv2
import gc
from PIL import Image

from paddleocr import PaddleOCR

c:\Users\lemer\anaconda3\envs\paddleocr\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fonction pour détection de dossart 

In [2]:
ocr = PaddleOCR(use_textline_orientation=True, lang='en')

c:\Users\lemer\anaconda3\envs\paddleocr\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lemer\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lemer\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\lemer\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', No

In [3]:
def detection_dossart(files, galerie_path): 

    start = time.time() 
    detection_results = []

    # parcours des images 
    for i, file in enumerate(files):
        print(f"Image : {file}")

        # redimensionnement de l'image 
        image_path = os.path.join(galerie_path, file)
        img = cv2.imread(image_path)

        # lecture de l'image si cv2 ne fonctionne pas 
        if img is None:
            try:
                pil_img = Image.open(image_path).convert("RGB")
                img = np.array(pil_img)
                img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            except Exception as e:
                print(f"Impossile de lire {image_path} : {e}")
                continue

        max_side = 2000
        height, width = img.shape[:2]
        scale = min(max_side/height, max_side/width, 1.0)
        if scale < 1.0:
            img = cv2.resize(img, (int(width*scale), int(height*scale)))
        
        # execution paddleOCR
        results = ocr.predict(img)

        # Si aucun texte n'est détecté 
        if not results or all(line is None for line in results):

            # créer une liste detection et ajout a detection_results
            detection = []
            detection.append(file)
            for _ in range(10):
                detection.append(None)
            detection_results.append(detection)
            continue

        # Si du texte est détecté 
        if results:
            detect_digit = False
            for page in results:
                texts = page['rec_texts']
                scores = page['rec_scores']
                polys = page['rec_polys']

                # Parcours de chaque couple text-score-coordonnées
                for text, score, poly in zip(texts, scores, polys):
                    # vérification qu'on détecte des nombres
                    if text.isdigit(): 
                        detect_digit = True
                        detection = []
                        detection.append(file)
                        detection.append(text)
                        detection.append(score)
                        detection.extend(poly.flatten().tolist())
                        detection_results.append(detection)

            if not detect_digit:
                # créer une liste detection et ajout a detection_results
                detection = []
                detection.append(file)
                for _ in range(10):
                    detection.append(None)
                detection_results.append(detection)

        # libération mémoire
        del results
        gc.collect()
        

    print(len(detection_results))

    end = time.time() 
    print(f"Time : {end - start}") 

    return detection_results

In [4]:
# récupération de toutes les galeries de photos
images_repertory = os.path.join('Images', '2025')
galeries = [d for d in os.listdir(images_repertory) if os.path.isdir(os.path.join(images_repertory, d))]

to_remove = ["10 km Moulin bleu", "10 km Rue des Sports", "10 km Source de l'Yvette", "21 km Source de l'Yvette", 
             "Autour de la course", "L'arrivée", "Podiums"]
for tr in to_remove:
    galeries.remove(tr)
print(galeries)

['100EOS5D', '101EOS5D']


In [5]:
for galerie in galeries:

    # Lister les fichiers image dans le dossier
    galerie_path = os.path.join(images_repertory, galerie)
    valids_extensions = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')
    files = sorted([f for f in os.listdir(galerie_path) if f.lower().endswith(valids_extensions)])
    
    # détection des dossarts
    detection_results = detection_dossart(files, galerie_path)

    # Sauvegarde des résultats

    # répertoire où sont sauvegardés les résultats
    folder_result = os.path.join("Results", "2025", galerie)
    os.makedirs(folder_result, exist_ok=True)
    result_file_name = "detection_v1.csv"
    result_file_path = os.path.join(folder_result, result_file_name)

    # Définition des en-têtes
    headers = ["file", "text", "score", "x1", "y1", "x2", "y2", "x3", "y3", "x4", "y4"]

    df = pd.DataFrame(detection_results, columns=headers)
    df.to_csv(result_file_path, index=False, encoding='utf-8')

    print(f'Fichier enregistré : {result_file_path}')
    print(os.listdir(folder_result))

Image : MU8A9394.JPG
Image : MU8A9395.JPG
Image : MU8A9396.JPG
Image : MU8A9397.JPG
Image : MU8A9398.JPG
Image : MU8A9401.JPG
Image : MU8A9402.JPG
Image : MU8A9403.JPG
Image : MU8A9404.JPG
Image : MU8A9405.JPG
Image : MU8A9406.JPG
Image : MU8A9407.JPG
Image : MU8A9409.JPG
Image : MU8A9410.JPG
Image : MU8A9411.JPG
Image : MU8A9412.JPG
Image : MU8A9413.JPG
Image : MU8A9414.JPG
Image : MU8A9415.JPG
Image : MU8A9416.JPG
Image : MU8A9417.JPG
Image : MU8A9418.JPG
Image : MU8A9419.JPG
Image : MU8A9420.JPG
Image : MU8A9421.JPG
Image : MU8A9422.JPG
Image : MU8A9423.JPG
Image : MU8A9424.JPG
Image : MU8A9425.JPG
Image : MU8A9426.JPG
Image : MU8A9427.JPG
Image : MU8A9428.JPG
Image : MU8A9429.JPG
Image : MU8A9430.JPG
Image : MU8A9432.JPG
Image : MU8A9433.JPG
Image : MU8A9434.JPG
Image : MU8A9435.JPG
Image : MU8A9436.JPG
Image : MU8A9437.JPG
Image : MU8A9438.JPG
Image : MU8A9439.JPG
Image : MU8A9440.JPG
Image : MU8A9441.JPG
Image : MU8A9442.JPG
Image : MU8A9443.JPG
Image : MU8A9444.JPG
Image : MU8A9

In [13]:
print(len(detection_results))

237
